# Logistic Regression - Predicting shipwreck survival

In this week's assignment you will implement a classification algorithm named Logistic Regression. This sounds very counter-intuitive, as regression is something entirely different from classification, but the Logistic Regression model is actually estimating the probability (which is a continuous value) that data can be assigned to a specific class out of a set of classes. We will focus on so-called binary classification problems, wherein there are two possible classes. Some of the examples of binary classification problems that Logistic Regression is able to solve are: 

- Email; spam or not spam 
- Online transactions; fraud or not fraud
- Tumor; malignant or benign 

In this notebook, we will be using the classic Titanic dataset. This data consists of demographic and traveling information for 891 of the Titanic's passengers, and the goal is to predict which of these passengers survived. 

Here is a summary of the data set's attributes:

- *PassengerId*: passenger ID assigned in this dataset
- *Survived*: a Boolean indicating whether the passenger survived or not (0 = No; 1 = Yes); this is our target
- *Pclass*: passenger class (1 = 1st; 2 = 2nd; 3 = 3rd)
- *Name*: field containing the name and title of the passenger
- *Sex*: male/female
- *Age*: age of the passenger in years
- *SibSp*: number of siblings/spouses aboard
- *Parch*: number of parents/children aboard
- *Ticket*: ticket number
- *Fare*: passenger fare in British Pounds
- *Cabin*: location of the cabin, consisting of a letter indicating the deck, and a cabin number.
- *Embarked*: port of embarkation (C = Cherbourg; Q = Queenstown; S = Southampton)

As you can probably see, the dataset contains a mix of textual, continuous, categorical, and boolean variables. Before we can get into implementing and applying Logistic Regression, we will have to clean up the data. We will lead you through this process using functions a data-scientist might use, providing you with the tools that will help you prepare your own data in the future.

Run the code below to load the data and display the first few rows of that data set.

In [ ]:
from notebook_checker import register_student, start_checks

%register_student
%start_checks

In [ ]:
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

data = pd.read_csv('titanic.csv')

display(data.head())

### A quick view

From the first five rows displayed in the table above, you might be able to see that there are multiple types of variables: we have integers, strings, floats, and a few *NaN*'s, which are missing values. Let's take a quick look to see what the types of data that we are dealing with actually are. A tool a data-scientist might use here is Panda's [`df.info()`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.info.html), which is a method that can be applied to any `DataFrame` that shows information about that frame, including the index, datatype for each column, non-null values, and memory usage.

In [ ]:
data.info()

As you can see we have 891 entries, numbered 0 to 890, with 12 different features. Of these 12 features, *Survived* will be our target features. Out of 12 columns, 2 are floats, 5 are integers, and 5 are "objects" which is the generic type, but in this case these are strings. Out of all of the columns, there are 3 that have some missing values. We will need to fix these missing values later.

Let's take a better look at what the numeric columns, i.e. those that contain integers and floats, look like. For this, we will use Panda's [`df.describe`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.describe.html?highlight=describe#pandas.DataFrame.describe), which is a method that can be applied to any `DataFrame` to show its descriptive statistics, which summarize the central tendency, dispersion and shape of a dataset’s distribution, excluding NaN values.

In [ ]:
display(data.describe())

The result is a table showing some statistics for each of the columns. The `count` is the number of values that are not NaN in that specific column. We can also see the mean, standard deviation, minimum, maximum, and percentiles for each of the columns. Percentiles are covered in *SOWISO* chapter 2 (Describing Data), the $50^{th}$ percentile corresponds to the median value. Let's see what we can conclude from this table and the previous one:

- *PassengerId* is a column that runs from 1 to 891, indicating the index of a passenger in this dataset. This variable has no descriptive value for the passenger, but it could still cause our machine learning model to find some correlations that are really just noise, so we should delete this whole column before using the data for training.
- The mean of *Survived* indicates that out of the 891 people in our dataset, only ~38% of people survived. This can be deduced from the mean of the column, because the value 1 corresponds to survived and 0 corresponds to did not survive. As an example, when 90% of these values would be 1, we would end up with a mean of 0.9, as the other values are all zeros.
- There are more people in *Passenger Class* 3 than in class 1 and 2 combined. This can be concluded from the fact that from the 50th percentile to the maximum value, every passenger is in third class.
- There is at least one baby in our dataset, with an *Age* of 0.42, or 5 months. The oldest person is 80 years old. We can see this from the min and max values of the *Age* column.
- There are 277 people of which we do not know the *Age*. We have 891 data entries, but the `count` for age is 714, so 891 - 714 = 277.
- The majority of people has no *Siblings/Spouses* aboard. This can be concluded from the percentiles.
- The majority of people has no *Parents/Children* aboard. This can be concluded from the percentiles.
- There is a big difference in scale in data. Fare can be up to 512 pounds while the number of *Parents/Children* on board is only up to 6.  We can also see this in the standard deviations, where the difference is also very large. In his videos, Andrew Ng talks about feature scaling, which is something we should apply here. We will discuss this later.

Often, these tables can tell you a lot about the data you are working with, and might provide you with intuitions on what would work best for this dataset. We will spend some more time on analysing the data at the end of this notebook.

### Assignment 1: Cleaning the data

First let's remove feature from the data that we won't use in the Logistic Regression model. Uniquely identifying features are generally a bad idea to include in training data, as they will never generalize well to new samples. In theory, the model might learn that a passenger with a specific ID is less likely to survive, but that is only relevant to the specific training data used and unlikely to improve predictions for other samples. In order to create a model that will generalize better, we'll completely remove *PassengerId*, *Name* and *Ticket* from this data set.

Create a new DataFrame called `clean_data` that is a direct copy of our original `data`, but does not include the columns *PassengerId*, *Name* and *Ticket*. Take a look at Panda's [`df.drop`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.drop.html) function, which can be used to delete columns from a given `DataFrame`, when setting the `axis` parameter correctly. The method returns a new `Dataframe` without the rows/columns that were deleted.

In [ ]:
# YOUR CODE HERE

clean_data.info()

If you did the step above correctly, the `clean_data` DataFrame should now only have 9 columns, as 3 were removed. Looking at this description, we can see there are quite a few missing values in the data set, which are indicated as *null* values. For example, the *Cabin* feature has $204$ non-null values, while the data set consists of $891$ entries, meaning there are $687$ values missing here.

One could say that the absence of information is also information, but when training a machine learning model, most models cannot make any predictions when one of the input variables is missing, making these samples unusable. When you encounter missing values, you have multiple options for dealing with them:

* You can delete each row (sample) that contains a missing value
* You can delete the whole column (feature) containing the missing values
* You can replace the missing values with some other value

Which of these you choose depends on the specific feature, the size of your data set and number of values missing. For the *Cabin* feature, the majority of the values are actually missing, which makes dropping the feature entirely a good solution. Note that we are still throwing away some potentially useful information here, as a cabin number might converted to information on where the passenger was on the ship, which in turn might give information on if a passenger would have been able to reach a lifeboat on time. In a more advanced data processing pipeline we might still include some of this data, but that would include data mining techniques that are beyond the scope of this notebook.

So, for now, we'll just remove the *Cabin* column completely, as for most of the samples it does not contain any information anyway. Again, use [`df.drop`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.drop.html), this time to remove the *Cabin* column from the `clean_data`. Remember to store the result into `clean_data` again, as the function returns the new DataFrame.


In [ ]:
# YOUR CODE HERE

clean_data.info()

Next, let's take a look the *Embarked* missing values. There appear to only be $2$ values missing here, as there are $889$ non-null values. The most common solution in such a case is to just remove those two samples from data set completely, as you'll only remove a few samples from your data, while retaining the *Embarked* information for most of them.

For this you can use the function [`df.dropna`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.dropna.html), which removes missing values from a DataFrame. As we only want to apply this solution to the *Embarked* column, make sure to use the optional argument `subset` here, which you can give a list of column names to include in the operation.

In [ ]:
# YOUR CODE HERE

clean_data.info()

The only remaining missing values should now be the $177$ passengers for which we do not know the age. Here, deleting the rows really isn't a great option, as we don't have a lot of data to start with, and we need as many training samples as possible to create an accurate model. Deleting the column also wouldn't be a good option, as there are many cases where we *do* have the age, and the age of a person will probably be a good indicator for whether a person survived or not. The third option, replacing the missing values, remains, but we have to find a method that will not skew our results. 

There are multiple ways of replacing missing values, but it is important to find a method that does not affect the outcome of our model too much. Replacing the values randomly will probably create a lot of noise and outliers, and therefore might prevent the model from learning any real correlations between age and survival of the passenger.  So, we will use a method that replaces missing values with values that are as generic as possible, as to not skew the data too much, by replacing all missing values with a value that is inferred from the data that is available. The easiest option for this is to just use the *mean* of all the non-missing values as the replacement value.

In the cell below, use Panda's [`df.fillna`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.fillna.html) function to replace every missing value in the *Age* column of `clean_data` with the average age based on all the non-missing values. You can use the [`mean()`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.mean.html) function on the *Age* column to compute the average age. Select the column *Age*, apply `fillna` to replace every missing value with the computed mean value and then store the result in the *Age* column to overwrite it.

In [ ]:
# YOUR CODE HERE

clean_data.info()
display(clean_data.describe())

In [ ]:
assert len(clean_data) == 889, "Something went wrong while cleaning the data. Did you remove the rows with no data for \'embarked\'?" 
assert not clean_data.isnull().any().any(), "Something went wrong while cleaning the data. There are still missing values in your dataframe."
print("Solution seems correct!")

As you can see, all missing values in the column *Age* have now been replaced with the average. This means that the average of the column has not changed. However, this solution is still not ideal. We have changed the standard deviation and the percentiles, which indicates that we have changed the entire distribution of the data. As there is no way to recover the real data, for now we will settle for this solution.

Finally, we will have to transform the columns *Sex* and *Embarked* from categorical data into a numerical type that the Logistic Regression can work with. The column *Sex* has two categories, male or female, while the port where passengers *Embarked* has three categories; C = Cherbourg, Q = Queenstown, or S = Southampton. In the previous module, you learned about one-hot encoding, which is exactly what we will use here too. Pandas has a method named [`get_dummies`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.get_dummies.html) that transforms data from categorical to a one-hot encoded set of columns.

Note that, for now, we will choose to interpret the *Passenger Class* as a numeric variable, as we won't one-hot encode it. However, you could make the argument that it is actually a categorical variable, as it only really has three values. In this case, both interpretations of the variable are valid, as it is a categorical variable with ordinal properties; second class is actually something "between" first and third class. Using the variable as if it is numerical, however, *does* imply that the difference between first and second class is the same as the difference between second and third class. Using the variable as if it is categorical implies that there is no order, and that the three different classes have no connection other than that you can only be part of one class.

In [ ]:
# Create two new columns for the male and female one-hot encoding
sex = pd.get_dummies(clean_data['Sex'], dtype=int)

# Create three new columns for port of embarkment one-hot encoding
embark = pd.get_dummies(clean_data['Embarked'], dtype=int)

# We no longer need these columns, as we will replace them soon
clean_data = clean_data.drop(['Sex','Embarked'], axis=1)

# Create the new dataframe with the one-hot columns we created from the categories in Sex and Embarked
clean_data = pd.concat([clean_data, sex, embark], axis=1)

clean_data.info()
display(clean_data.head())

### Final preparations

Now that all data is numerical, we can continue to the next step in processing the data. Earlier, we already observed that there is a big difference in the scale of each of the features in the data, which can impact the learning process for a model (see the theory video for more details). Feature scaling can be used to ensure that the different features in your dataset take on similar ranges of values.

Scipy's `stat` module has some interesting methods to do this, including the `zscore` method which you might also recognise from the theory video:

$$ \tilde{\mathbf{x}}_j = \frac{\mathbf{x}_j - \mu_j}{\sigma_j} $$

Here $\mathbf{x}_j$ is a vector of x-values for the $j^{th}$ column, $\mu_j$ and $\sigma_j$ are the (scalar) mean and standard deviation for that same column, and $\tilde{\mathbf{x}}_j$ is the scaled version of the feature vector. The basic principle here is that it transforms every value in the column $j$ to a value that indicates how many standard deviations the value originally was away from the mean of the data *in that column*. For example: a $Z$-score of $-2$ would indicate that the value was 2 standard deviations below the average, while a $Z$-score of $0$ would indicate that the value was actually equal to the mean of the column.

In the code below, we transform each of the original numerical data columns using the `zscore` method, by applying it to each of the elements in those columns. The function `zscore` conveniently takes care of computing all means and standard deviations for each column separately, and returns the scaled data.

Note that we do not want to apply this function to our newly one-hot encoded data, as those are already within a nice $[0,1]$ range and scaling these binary values by their mean and standard deviation wouldn't really make sense here.


In [ ]:
from scipy.stats import zscore

numerical = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
clean_data[numerical] = clean_data[numerical].apply(zscore)

display(clean_data.describe())

The columns that we have adjusted with our `zscore` function now all have a mean that is very close to zero, and a standard deviation that is very close to one. So, the resulting data is now sufficiently of equal scale, and mean-centered.

With the data completely cleaned up, the final step is to put the data into format useful for learning. This means splitting the data into the input features `X` and the target variable `y`. In addition, we'll convert the data to *Numpy* `ndarrays` here, so we can again easily use vectorization in our Logistic Regression implementation. We've used Pandas for reading the data and cleaning it up (which is where *Pandas* excels), and now we'll use *Numpy* for the actual learning / computation portion. Finally, we'll need to also split the data set into a training and test set for learning, for which we'll use the `sklearn` function `train_test_split`.

In [ ]:
from sklearn.model_selection import train_test_split

# Split the data into target "y" and input "X"
y_df = clean_data['Survived']
X_df = clean_data.drop('Survived', axis=1)

# Convert the Pandas DataFrames to Numpy ndarrays
X_np = X_df.to_numpy()
y_np = np.expand_dims(y_df.to_numpy(), axis=1)

#Split the data into 70% training and 30% testing
X_train, X_test, y_train, y_test = train_test_split(X_np, y_np, train_size=0.7, random_state=1265599650)

### Assignment 2: The Logistic function

For Logistic Regression, we require a function that generates probabilities; a function that gives outputs between 0 and 1 for **all** inputs. There are many functions that meet this description, but the one that is used in logistic regression is the logistic function:

$$ g(z) = \frac{1}{1+e^{-z}} $$

Implement the function `logistic_func` that can either take a single value `z`, or an array of values `z`, and compute the *Logistic* function for each.

*Hint:* Numpy built-in functions and basic arithmetic operations work on both Numpy arrays and single values. When the function detects that its input is a Numpy array, the operation it applies will be applied to each element in the array separately, the result being a Numpy array of the same shape as the input. This is called *element-wise application*, and many mathematical operations have Numpy versions that support element-wise application. Write your `logistic_func` in such a way that it can compute $g$ values for scalars or for entire vectors. You can search for relevant *Numpy* functions to use in this list of [mathematical functions](https://numpy.org/doc/stable/reference/routines.math.html).

Then, write some code that can plot the results for the logistic function between $-5$ and $5$. Use Numpy's [`np.linspace`](https://numpy.org/doc/stable/reference/generated/numpy.linspace.html?highlight=linspace#numpy.linspace) to generate a range of $z$ values and use your `logistic_func` to calculate the corresponding $g$ values. Make sure the resulting plot looks correct, before moving on to the next step.

In [ ]:
def logistic_func(z):
    # YOUR CODE HERE
    
    
# YOUR CODE HERE

### Assignment 3: Building the Logistic model

Next, we can start working on the model function, which will generate a vector of class probabilities, given some matrix of inputs $X$, a vector of weights $\mathbf{w}$ and a bias $b$. For logistic regression, we'll use the logistic function $g$ to transform the hypothesis we used in multivariate linear regression into a hypothesis function that only generates probabilities (i.e. results between 0 and 1). For each of the $m$ samples, the prediction $\hat{y}$ is computed as:

$$ \hat{y}^{(1)} = g(\mathbf{x}^{(1)} \cdot \mathbf{w} + b)$$
$$ \hat{y}^{(2)} = g(\mathbf{x}^{(2)} \cdot \mathbf{w} + b)$$
$$ \hat{y}^{(3)} = g(\mathbf{x}^{(3)} \cdot \mathbf{w} + b)$$
$$\dots$$
$$ \hat{y}^{(m)} = g(\mathbf{x}^{(m)} \cdot \mathbf{w} + b)$$

This is exactly the same hypothesis equation as for multivariate linear regression, just with the logistic function $g$ applied as the final step. Here the function $g$ is the Logistic function defined above. As you've just written `logistic_func` specifically to also work on vectors, if we construct a vector of $z$ values for all samples at once using a matrix multiplication, then we could also write the entire hypothesis function in a vectorized form:

$$
\left[\begin{array}{c} z^{(1)} \\ z^{(2)} \\ z^{(3)} \\
\vdots \\ z^{(m)} \end{array} \right]
=
\left[\begin{array}{cccc}
x_1^{(1)} & x_2^{(1)} & x_3^{(1)} & \cdots & x_n^{(1)} \\ 
x_1^{(2)} & x_2^{(2)} & x_3^{(2)} & \cdots & x_n^{(2)} \\ 
x_1^{(3)} & x_2^{(3)} & x_3^{(3)} & \cdots & x_n^{(3)} \\ 
\vdots & \vdots & \vdots & \ddots & \vdots \\
x_1^{(m)} & x_2^{(m)} & x_3^{(m)} & \cdots & x_n^{(m)} \\ 
\end{array} \right]
\left[\begin{array}{c} w_1 \\ w_2 \\ w_3 \\ \vdots \\ w_n \end{array} \right]
+
\left[\begin{array}{c} b \\ b \\ b \\ \vdots \\ b \end{array} \right]
$$

Next, we can apply the Logistic function to each of the elements of this $\mathbf{z}$ vector:

$$
\left[\begin{array}{c}
\hat{y}^{(1)}\\
\hat{y}^{(2)}\\
\hat{y}^{(3)}\\
\vdots \\
\hat{y}^{(m)} \\
\end{array} \right]
=
\left[\begin{array}{c}
g(z^{(1)})\\
g(z^{(2)})\\
g(z^{(3)})\\
\vdots \\
g(z^{(m)}) \\
\end{array} \right]
$$

Large parts of these equations should look familiar, as they're really just *Multivariate Linear Regression* with an extra *Logistic function* applied. This means you can (and should) take inspiration from your multivariate linear solutions, although some of the details will be a little different here. Make sure to also use *broadcasting* to add the scalar bias $b$ directly to result of the matrix multiplication (so no need to construct a whole vector of b's).

Write the function `logistic_model` to compute the hypothesis vector of the model for some matrix `X`, a parameter vector `w`, and a bias term `b`. Your solution for `logistic_model` should rely on `logistic_func` working for vectors as well as scalars, and so you shouldn't need any loops here at all.

In [ ]:
def logistic_model(X, w, b):
    # YOUR CODE HERE



### Assignment 4: The Cost function

Now that we can make predictions using an input matrix $X$, the parameter vector $\mathbf{w}$ and bias term $b$, we can evaluate the cost of our model. To be able to apply gradient descent, we will need a cost function that is convex when we use our logistic model to make predictions. This cost function for logistic regression is explained in detail in the theory videos, and is defined as:

$$J(\mathbf{w}, b) = - \frac{1}{m} \sum_{i=1}^m y^{(i)} log(\hat{y}^{(i)}) + (1 - y^{(i)}) log(1 - \hat{y}^{(i)})$$

Implement the function `logistic_cost`. Use your `logistic_model` function to calculate the prediction vector $\mathbf{\hat{y}}$ and the compute the cost for that prediction using only *Numpy* operations, so without using any loops.

*Hints:* Try to rewrite this equation slightly, such that it can be implemented using just two matrix multiplications or dot products. Recall that, when using *Numpy* arrays, all basic arithmetic operations (i.e. `+`, `-`, `*` and `/`) are done *elementwise*, and will be *broadcasted* if needed.

In [ ]:
def logistic_cost(w, b, X, y):
    # YOUR CODE HERE
    


In [ ]:
assert np.allclose(logistic_cost(np.ones((X_train.shape[1], 1)), 1, X_train, y_train), 2.013825365789087), 'Cost for training data seems incorrect'
assert np.ndim(logistic_cost(np.ones((X_train.shape[1], 1)), 1, X_train, y_train)) == 0, 'Result seems to be a matrix and not a scalar. Did you use np.squeeze?'
print("Solution seems correct!")

### Assignment 5: Obtaining the Gradient terms for Logistic Regression

Manually taking the whole partial derivative of this cost function can be a little time consuming, but in order to still practice this skill, we'll stilll do a part of this derivative. For this part of the notebook, you should consult the `logistic_derivative.pdf` supplement, included as a part of the assignment. Read sections 1, 2 and 3 in the supplement, and then complete the steps of the partial derivative below, labeling each of the steps with their respective rules (as with the linear regression assignment).

$$\frac{\partial L^{(i)}}{\partial \hat{y}^{(i)}} =
\frac{\partial}{\partial \hat{y}^{(i)}} \bigl(-y^{(i)} log(\hat{y}^{(i)})
- (1 - y^{(i)}) log(1 - \hat{y}^{(i)}) \bigr)$$

**TODO:** *Your steps, each labeled with what rules you've applied, should go here*.

$$\frac{\partial L^{(i)}}{\partial \hat{y}^{(i)}} =
\frac{-y^{(i)}}{\hat{y}^{(i)}} + \frac{1 - y^{(i)}}{1 - \hat{y}^{(i)}}$$

Section 4 of the supplement is optional, but definitely read sections 5, 6, and 7 before moving on to the next step. 

### Assignment 6: Gradients and Gradient Descent

In the end, the partial derivatives for $\mathbf{w}$ and $b$ end up looking exactly the same as for linear regression. While the model function to compute the prediction $\mathbf{\hat{y}}$ does still change, the rest of the derivatives remain the same, even though the cost function is completely different.

As shown in the theory videos, and section 7 of the supplement, the gradient for $b$ is just

$$\frac{\partial J}{\partial b} = \frac{1}{m} \sum_{i=1}^m (\hat{y}^{(i)} - y^{(i)})$$

where $\hat{y}^{(i)}$ is the prediction made for the $i^{th}$ sample by the logistic regression model function

$$\begin{align*}
\hat{y}^{(i)} &= g(\mathbf{x}^{(i)} \cdot \mathbf{w} + b) \\
&= f_{\mathbf{w},b}(\mathbf{x}^{(i)})
\end{align*}$$

For the parameter vector $\mathbf{w}$, the derivatives for the gradient are

$$\frac{\partial J}{\partial w_1} = \frac{1}{m} \sum_{i=1}^m (\hat{y}^{(i)} - y^{(i)})x^{(i)}_1$$
$$\frac{\partial J}{\partial w_2} = \frac{1}{m} \sum_{i=1}^m (\hat{y}^{(i)} - y^{(i)})x^{(i)}_2$$
$$\dots$$
$$\frac{\partial J}{\partial w_n} = \frac{1}{m} \sum_{i=1}^m (\hat{y}^{(i)} - y^{(i)})x^{(i)}_n$$

which can be combined together into a gradient vector for $\mathbf{w}$

$$\frac{\partial J}{\partial \mathbf{w}} = \left[\begin{array}{c} \frac{\partial J}{\partial w_1} \\ \frac{\partial J}{\partial w_2} \\ \vdots \\ \frac{\partial J}{\partial w_n} \end{array} \right]$$


Write the functions `b_gradient`, `w_gradient` and `gradient_descent` for this *Logistic Regression* model. You should be able to reuse a lot of your code from *Multivariate Linear Regression* here, as a large part of this code actually hasn't changed!

In [ ]:
def b_gradient(w, b, X, y):
    # YOUR CODE HERE
    
    
def w_gradient(w, b, X, y):
    # YOUR CODE HERE
    
    
def gradient_descent(X, y, w, b, alpha, thres=10**-6):
    # YOUR CODE HERE
    

# Initialize w to a zero vector of the correct shape, and b to zero
w = np.zeros((X_train.shape[1], 1))
b = 0

# Find the w vector that minimizes the cost function
w_hat, b_hat = gradient_descent(X_train, y_train, w, b, 10**-4)

print('w:', w_hat)
print('b:', b_hat)

In [ ]:
assert np.allclose(w_hat, np.array([[-0.42245195], [-0.16952217], [-0.11465412], [0.08995297], [0.24693307], [0.47087209], [-0.64376821], [0.05535226], [0.00347353], [-0.2317219]])), 'Optimal w vector for training data seems incorrect'
assert np.allclose(b_hat, -0.17289611), 'Optimal b for training data seems incorrect'
print("Solution seems correct!")

### Assignment 7: Inspecting the learning curves

While this model probably converges with the provided parameters, it is hard to tell if gradient descent actually converged to a *good* solution. In order to assess this, we'll make a modified version of gradient descent that plots the learning curves. Learning curves show how the cost changes with each step of gradient descent, and plotting them can tell you a lot about how well an algorithm converged.

You can plot the learning curve just for the training cost, where ideally you'd see the cost go down for each step (otherwise the algorithm is diverging) and eventually the cost would level off to almost flat, indicating that gradient descent cannot improve the parameters much further. You can also measure the testing or the validation cost at each step and see if this follows the same trend down as the training cost. If the training cost is going down, but your validation cost is going up, then your model must be overfitting.

Complete the function `gradient_descent_learning_curves`, which should implement exactly the same gradient descent algorithm as in the previous assignment. In addition, it should store the training and testing cost at every step of the algorithm, and store the number of iterations gradient descent took to converge. Once the algorithm is converged, the function should plot the training and the testing cost as a function of the number of iterations, clearly labeling which line in the figure belongs to which cost. Finally, the function should return the converged result for `w` and `b`, as before.

The function will need a few additional arguments to determine the testing cost, namely `X_test` and `y_test`, so you should modify the function call accordingly. For this function call, you should also tweak the learning rate `alpha` and the convergence threshold `thres` and inspect the learning curves each time. Repeat this process and make sure you find a configuration you think has actually converged to a good solution. Make sure to store the resulting optimized parameters as `w_hat` and `b_hat` (so the print at the end functions correctly).


In [ ]:
def gradient_descent_learning_curves(X_train, y_train, X_test, y_test, w, b, alpha, thres):
    # YOUR CODE HERE


w = np.zeros((X_train.shape[1], 1))
b = 0
    
# YOUR CODE HERE

print('w:', w_hat)
print('b:', b_hat)

### Collaborative Questions

The next two open questions are a little harder then the usual open questions, so for these you can, and should, confer with a fellow student on what you think the answers should be. As a result, similar answers will of course not be a problem for these question, just make sure to also mention with which students you discussed them.

**CQ1. Explain which values you chose for the learning rate and convergence threshold of gradient descent. What happened to the learning curve when you changed these values?**

*Your answer goes here.*

**CQ2. For this model, it is actually not possible to overfit, no matter how much you tweak the learning rate and convergence threshold. Explain why a Logistic Regression model with these inputs could never overfit the data.**

*Your answer goes here.*

### Assignment 8: Predictions

Now that we have a learned good values to use for the parameters $\mathbf{w}$ and $b$ on the training samples, we can use these to make predictions for our test samples `X_test`. The learned model function then actually estimates the probability of the passengers in `X_test` having survived the shipwreck or not.

$$P(y^{(i)}=1|x^{(i)};\mathbf{w},b) = f_{\mathbf{w},b}(\mathbf{x}^{(i)})$$

Finally, these estimates of the probability that $y=1$ still need to be transformed into boolean predictions, which will be the actual classifications. In the end, this regression model should still predict that either the person has survived $y=1$, or that the person has not survived $y=0$.

Write the function `predict`, that takes a matrix of input values `X`, a parameter vector `w` and bias term `b`, and transforms them into a vector of boolean predictions. Use `logistic_model` to generate predictions, and then use a decision boundary of $f_{\mathbf{w},b}(\mathbf{x}) \geq 0.5$ to determine when you should classify each sample as $y=1$.

In [ ]:
def predict(X, w, b):
    # YOUR CODE HERE
    

predictions_train = predict(X_train, w_hat, b_hat)
predictions_test = predict(X_test, w_hat, b_hat)

### Assignment 9: Determining accuracy

Next, we would like to see how *accurate* our model is, now that we have trained it and it can generate predictions. Implement the function `calc_accuracy` that accepts a vector of `predictions` and a vector of *ground-truth* values `y`. The function should return the ratio of correct predictions (predictions where the value in `predictions` is equal to the value in `y`). The ideal situation where every classification is correct, should result in an accuracy of $1$.

In [ ]:
def calc_accuracy(predictions, y):
    # YOUR CODE HERE
    

print(f"Training set accuracy: {calc_accuracy(predictions_train, y_train)}")
print(f"Testing set accuracy: {calc_accuracy(predictions_test, y_test)}")

In [ ]:
assert np.allclose(calc_accuracy(predict(X_test, np.array([[-0.42245195], [-0.16952217], [-0.11465412], [0.08995297], [0.24693307], [0.47087209], [-0.64376821], [0.05535226], [0.00347353], [-0.2317219]]), -0.17289611), y_test), 0.7565543071161048), 'Accuracy for testing data seems incorrect'
print("Solution seems correct!")

### Assignment 10: Confusion matrix

The accuracy we have just determined is, of course, one of the primary goals of building a model; we generally want a Machine Learning model to be as accurate as possible. However, it does not give much insight into in what way our model was accurate. As an example, let's say that we want to diagnose whether a patient has a virus. Of course we want our model to diagnose as accurately as possible, but there are two types of error the model could make:

1. A patient *has* the virus, but was diagnosed as being negative
2. A patient *does not have* the virus, but was diagnosed as being positive

One of these types of mistakes is potentially *much* more costly than the other, as falsely diagnosing a patient as negative could become very harmful for the patient and their environment. Although an extreme (and currently very relevant) example, it is quite often the case in classification that not every error has the same associated *cost*.

To give better insight into the performance of a classification algorithm, typically, a confusion matrix is made. The confusion matrix is a table layout that allows quick identification of the performance of a classification algorithm. Each *row* in the matrix represents the data that got predicted as a specific class, while each *column* represents the data that is actually in a class:

<table>
    <thead>
        <tr> <th colspan=2></th><th colspan=2 style="border: 1px solid black;"> Actual class </th> </tr>
        <tr>
            <th colspan=2></th>
            <th style="border: 1px solid black;">P</th>
            <th style="border: 1px solid black;">N</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td rowspan=2 style="border: 1px solid black;"><b> Predicted class </b></td>
            <td style="border: 1px solid black;"><b> P </b></td>
            <td style="border: 1px solid black;"><b> TP </b></td>
            <td style="border: 1px solid black;"> FP </td>
        </tr>
        <tr>
            <td style="border: 1px solid black;"><b> N </b></td>
            <td style="border: 1px solid black;">FN</td>
            <td style="border: 1px solid black;"><b> TN </b></td>
        </tr>
    </tbody>
</table>

Where:
- P = Positive
- N = Negative
- TP = True Positive, denoting every value that was correctly predicted as positive
- FP = False Positive, denoting every value that was predicted as positive but was actually negative
- TN = True Negative, denoting every value that was correctly predicted as negative
- FN = False Negative, denoting every value that was predicted as negative but was actually positive

For example, using the example of diagnosis of patients, let's say that 100 people take a test, and of these people, 80 actually have the virus. We have two different testing kits that are used to diagnose the patients, and these are the resulting confusion matrices:


##### Testing kit 1
<table>
    <thead>
        <tr> <th colspan=2></th><th colspan=2 style="border: 1px solid black;"> Actual class </th> </tr>
        <tr>
            <th colspan=2></th>
            <th style="border: 1px solid black;">P</th>
            <th style="border: 1px solid black;">N</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td rowspan=2 style="border: 1px solid black;"><b> Predicted class </b></td>
            <td style="border: 1px solid black;"><b> P </b></td>
            <td style="border: 1px solid black;"><b> 56 </b></td>
            <td style="border: 1px solid black;"> 1 </td>
        </tr>
        <tr>
            <td style="border: 1px solid black;"><b> N </b></td>
            <td style="border: 1px solid black;"> 24 </td>
            <td style="border: 1px solid black;"><b> 19 </b></td>
        </tr>
    </tbody>
</table>

##### Testing kit 2
<table>
    <thead>
        <tr> <th colspan=2></th><th colspan=2 style="border: 1px solid black;"> Actual class </th> </tr>
        <tr>
            <th colspan=2></th>
            <th style="border: 1px solid black;">P</th>
            <th style="border: 1px solid black;">N</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td rowspan=2 style="border: 1px solid black;"><b> Predicted class </b></td>
            <td style="border: 1px solid black;"><b> P </b></td>
            <td style="border: 1px solid black;"><b> 62 </b></td>
            <td style="border: 1px solid black;"> 10 </td>
        </tr>
        <tr>
            <td style="border: 1px solid black;"><b> N </b></td>
            <td style="border: 1px solid black;"> 18 </td>
            <td style="border: 1px solid black;"><b> 10 </b></td>
        </tr>
    </tbody>
</table>

Now, we can learn a lot from these confusion matrices. First, the calculated accuracy of our first virus diagnosis testing kit is 77%, as 56 people are correctly diagnosed as positive, and 19 people are correctly diagnosed as negative.  The calculated accuracy for the second testing kit is 72%. However, using the first testing kit 24 people have been incorrectly diagnosed as negative, while for the second this was 18. Now, arguably, we would prefer to use testing kit 2, as even though the total accuracy of this testing kit is lower, this kit produces fewer (harmful) false negatives. 

Implement the function `confusion_matrix` that, given `predictions` and truth values `y`, creates a $2 \times 2$ Numpy matrix that contains the number of TP, FP, FN, and TN for these predictions.

In [ ]:
def confusion_matrix(predictions, y):
    # YOUR CODE HERE
    

print(confusion_matrix(predictions_test, y_test))

**Q1. Which type of error is larger for the Logistic Regression model on this Titanic data set; false positives or false negatives? Explain your answer.**

*Your answer goes here.*

### Correlations and understanding weights

To better understand what relations our model has found in the data we will have to take a look at the weights that our model has learned. The relative size of each weight is a direct indication of how important the corresponding feature is in defining the relationship between the input features and the target output. This is because the model function is entirely dependant on the linear combination of the input values and our weights:

$$\begin{align*}
f_{\mathbf{w},b}(\mathbf{x}^{(i)}) &= g(\mathbf{x}^{(i)} \cdot \mathbf{w} + b)\\
&= g(x^{(i)}_1 w_1 + x^{(i)}_2 w_2 + \dots + x^{(i)}_n w_n + b)
\end{align*}$$

The logistic function simply "squashes" the combined positive values to a maximum of 1, and the combined negative values to a minimum of 0. A negative weight therefore means that a large value for the corresponding feature will actually decrease the chances of survival for a passenger, while a postive weight means the corresponding feature has positive impact on the survival chance of a passenger.

The input for our function $g$ is composed out of the sums of each input multiplied with its corresponding weight. Since we have made sure that our data all approximately has equal scale by using the `zscore` method, bigger values for a specific weight also mean in the feature corresponding to that weight has a larger influence on the model prediction.

Below, we have used Seaborn to create a barplot that shows each of the weights and their corresponding data column names.

In [ ]:
sns.barplot(x=X_df.columns, y=np.squeeze(w_hat))
plt.show()

This plot shows that quite a few features got a non-zero weight, indicating they either contributed positively or negatively to the predicted survival chance of a passenger. The three most important predictive features seem to be the passenger class, the age of the passenger and whether the passenger was male or female. We cannot directly draw conclusions about the relationship between these features and whether or not a person survived based on this plot, but we *can* use it as a starting point for further analysis.

## Analyzing the data

First, let's take a look at a basic count plot to see how many people have survived overall.

In [ ]:
sns.countplot(x='Survived',data=data)
plt.show()

We can see from this plot that approximately 350 people have survived, while approximately 550 people have died overall.

Now, let's take a look at those 3 features specifically, and see how they affect the distribution.

### 1. Gender of the passenger

In [ ]:
sns.countplot(x='Survived',hue='Sex',data=data)
plt.show()

### 2. Class of the passenger's cabin

In [ ]:
sns.countplot(x='Survived',hue='Pclass',data=data)
plt.show()

### 3. Age of the passenger

This is a continuous variable, so we can't plot all the distinct cases. Instead we'll use a barplot with the averages for both classes, and corresponding error bars.

In [ ]:
sns.barplot(x='Survived', y='Age', data=data)
plt.show()

## Creating hypotheses for the data

For each of the 3 features plotted above, compare their individual impact on the survival chance of a passenger. For every feature, explain if the this feature has a positive or negative effect on the survival chance of a passenger and give your own hypothesis on why this feature is likely to effect survival (you may speculate here). Finally, relate this conclusion back to the positive or negative weight the feature gets in the learned logistic regression model above:

**Q2. Gender of the passenger**

*Your answer goes here.*

**Q3. Class of the passenger's cabin**

*Your answer goes here.*

**Q4. Age of the passenger**

*Your answer goes here.*